# Project 3.1 — Oscillators
**Topic:** Numerical integration of anharmonic oscillators using RK4. Phase-space analysis, energy conservation, and the difference between linear and nonlinear dynamics.


---
## 1. Set Up

### Physical Motivation
An oscillator with potential $U(x) = k|x|^\alpha / \alpha$ generalises the harmonic oscillator ($\alpha=2$). The equations of motion:

$$\ddot{x} = -k \cdot \text{sgn}(x) \cdot |x|^{\alpha-1} - b\dot{x} + A\cos(\omega_d t)$$

Each term has physical meaning:
- $-k|x|^{\alpha-1}\text{sgn}(x)$: **restoring force** from potential $U$. For $\alpha < 2$: soft spring (force weaker than harmonic); $\alpha > 2$: hard spring (stronger than harmonic).
- $-b\dot{x}$: **viscous damping** — linear in velocity, models fluid resistance at low Reynolds number.
- $A\cos(\omega_d t)$: **periodic driving** — can lead to resonance, period-doubling, or chaos.

### Why RK4?
The simple Euler method has **first-order** local error $\mathcal{O}(\Delta t^2)$ and global error $\mathcal{O}(\Delta t)$. For oscillators, Euler causes **artificial energy growth** over many periods — the trajectory spirals outward even with no driving. RK4 has local error $\mathcal{O}(\Delta t^5)$ and global error $\mathcal{O}(\Delta t^4)$, dramatically better. However, RK4 is **not symplectic** — it does not exactly conserve phase-space volume (Liouville's theorem) and will show slow energy drift for very long integrations.

### ⚠️ Critical Limitation: RK4 is NOT Symplectic
For conservative Hamiltonian systems, the correct long-time integrator is a **symplectic method** (e.g. Störmer–Verlet / leapfrog). These conserve a modified Hamiltonian exactly and maintain bounded energy error for arbitrarily long times. RK4's energy drift is small but secular — it grows without bound.


In [ ]:
import os, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

OUT = Path("output_oscillators")
OUT.mkdir(exist_ok=True)
np.set_printoptions(precision=6, suppress=True)
print(f"Output directory: {OUT.resolve()}")


---
## 2. Functions

### `rk4_step(f, t, y, dt, params)`
The 4th-order Runge-Kutta step. The key idea: **Simpson's-rule weighted average** of four slope estimates:

$$y_{n+1} = y_n + \frac{\Delta t}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

where:
- $k_1 = f(t_n, y_n)$ — slope at the start
- $k_2 = f(t_n + \Delta t/2,\; y_n + \Delta t/2 \cdot k_1)$ — slope at midpoint using Euler prediction
- $k_3 = f(t_n + \Delta t/2,\; y_n + \Delta t/2 \cdot k_2)$ — corrected midpoint slope
- $k_4 = f(t_n + \Delta t,\; y_n + \Delta t \cdot k_3)$ — slope at end using corrected midpoint

The weights $1:2:2:1$ come from Simpson's quadrature rule applied to the integral $\int_{t_n}^{t_{n+1}} f\,dt$.

**Why does it take 4 function evaluations?** Because achieving $\mathcal{O}(\Delta t^5)$ local error requires matching 4 terms in the Taylor expansion of $y(t_{n+1})$.

### `oscillator_rhs(t, y, p)` — State Vector Convention
State $y = [x, v]$ where $v = \dot{x}$. This converts the **second-order ODE** $\ddot{x} = F(x, \dot{x}, t)$ into a **first-order system**:

$$\frac{d}{dt}\begin{pmatrix}x\\v\end{pmatrix} = \begin{pmatrix}v\\F(x,v,t)\end{pmatrix}$$

This is the standard reduction of order. RK4 operates on this 2-component vector.

```python
power = power_right if x >= 0 else power_left
```
Allows **asymmetric potentials** where the left and right arms of the oscillator have different stiffness exponents. Physically: a bouncing ball on a hard floor (hard on the left, soft on the right).

```python
spring = -k * np.sign(x) * abs(x)**(power - 1.0)
```
Restoring force: $F = -dU/dx = -k|x|^{\alpha-1}\text{sgn}(x)$. The `np.sign` ensures correct direction for both $x>0$ and $x<0$. The `abs(x)` prevents `NaN` when $x<0$ and `power < 1`.

```python
if abs(x) == 0: spring = 0.0
```
Guards against $0^{(\alpha-1)}$ for $\alpha < 1$. **Problem**: `abs(x) == 0` is an exact floating-point comparison — almost never true for continuous integration. Better: `if abs(x) < 1e-300: spring = 0.0`.

### 🔧 Improvement Directions
1. **Symplectic integrator (Störmer-Verlet)**: for pure conservative oscillators, replace RK4 with:
   $x_{n+1} = x_n + v_n\Delta t + \frac{1}{2}a_n\Delta t^2$,
   $v_{n+1} = v_n + \frac{1}{2}(a_n + a_{n+1})\Delta t$.
   This conserves energy to machine precision over arbitrary time.
2. **Poincaré sections**: for driven oscillators, sample $(x, v)$ once per driving period. Fixed points in Poincaré section indicate period-1 orbits; a cloud indicates chaos.
3. **Bifurcation diagram**: vary $A$ and plot the long-time values of $x(nT)$. Period-doubling cascade visible before onset of chaos.
4. **Adaptive step size**: use step doubling or embedded RK4-RK5 (Dormand-Prince) to control local error automatically.


In [ ]:
def rk4_step(f, t, y, dt, params):
    k1 = f(t,             y,                 params)
    k2 = f(t + 0.5*dt,   y + 0.5*dt*k1,    params)
    k3 = f(t + 0.5*dt,   y + 0.5*dt*k2,    params)
    k4 = f(t + dt,        y + dt*k3,         params)
    return y + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

def oscillator_rhs(t, y, p):
    x, v = y
    k     = p.get("k", 1.0)
    power = p.get("power_right", p.get("power", 2.0)) if x >= 0             else p.get("power_left", p.get("power", 2.0))

    spring = 0.0 if abs(x) < 1e-300 else -k * np.sign(x) * abs(x)**(power - 1.0)

    b      = p.get("b", 0.0)
    gamma  = p.get("gamma", 0.0)
    n_drag = p.get("n_drag", 1.0)
    drive  = p.get("drive_amplitude", 0.0) * np.cos(p.get("drive_omega", 1.0)*t)

    force  = spring - b*v - gamma*abs(v)**(n_drag-1.0)*v + drive
    return np.array([v, force / p.get("mass", 1.0)])

def integrate_oscillator(y0, t_max=80.0, dt=0.01, params=None):
    if params is None:
        params = {}
    t = np.arange(0.0, t_max + 0.5*dt, dt)
    y = np.empty((t.size, 2), dtype=float)
    y[0] = y0
    for j in range(1, t.size):
        y[j] = rk4_step(oscillator_rhs, t[j-1], y[j-1], dt, params)
    return t, y[:, 0], y[:, 1]

def energy_symmetric(x, v, mass=1.0, k=1.0, power=2.0):
    return 0.5*mass*v**2 + k*np.abs(x)**power / power    # T + U


---
## 3 & 4. Calculation & Writeouts

### Cases to Study
1. **Linear harmonic** ($\alpha=2$, no damping, no drive): period $T = 2\pi/\sqrt{k/m}$. Perfect sinusoidal motion. Energy exactly conserved (RK4 error $\sim 10^{-8}$ per period for $\Delta t = 0.01$).
2. **Soft spring** ($\alpha < 2$): period increases as amplitude increases. Phase portrait orbits are "squashed" — wider in $x$ than the harmonic case.
3. **Hard spring** ($\alpha > 2$): period decreases as amplitude increases. Relevant to molecular bonding (Morse potential goes from harmonic to anharmonic).
4. **Damped**: spiral inward in phase space toward the fixed point $(0,0)$.
5. **Driven resonance**: when $\omega_d \approx \omega_0 = \sqrt{k}$, amplitude grows until limited by damping.


In [ ]:
cases = [
    ("Harmonic (α=2)",     {"k":1.0,"power":2.0},                     [1.0, 0.0]),
    ("Soft spring (α=1.5)",{"k":1.0,"power":1.5},                     [1.0, 0.0]),
    ("Hard spring (α=3)",  {"k":1.0,"power":3.0},                     [1.0, 0.0]),
    ("Damped harmonic",    {"k":1.0,"power":2.0,"b":0.15},            [1.5, 0.0]),
    ("Driven near res.",   {"k":1.0,"power":2.0,"b":0.1,
                            "drive_amplitude":0.5,"drive_omega":0.98}, [0.0, 0.0]),
]

fig, axes = plt.subplots(len(cases), 3, figsize=(15, 4*len(cases)))

for i, (label, params, y0) in enumerate(cases):
    t, x, v = integrate_oscillator(y0, t_max=60.0, dt=0.01, params=params)
    E = energy_symmetric(x, v, k=params.get("k",1.0),
                         power=params.get("power",2.0))
    E_rel = (E - E[0]) / (E[0] + 1e-12)          # relative energy drift

    # x(t)
    axes[i,0].plot(t, x, lw=1.0, color='steelblue')
    axes[i,0].set_ylabel('$x(t)$')
    axes[i,0].set_title(label)

    # Phase portrait
    axes[i,1].plot(x, v, lw=0.6, color='tomato', alpha=0.8)
    axes[i,1].set_xlabel('$x$'); axes[i,1].set_ylabel('$v$')
    axes[i,1].set_title('Phase portrait')
    axes[i,1].plot(x[0], v[0], 'go', ms=6)   # start point

    # Energy drift
    axes[i,2].plot(t, E_rel, lw=1.0, color='darkgreen')
    axes[i,2].set_xlabel('$t$')
    axes[i,2].set_ylabel('$(E-E_0)/E_0$')
    axes[i,2].set_title('Relative energy drift (RK4)')
    axes[i,2].axhline(0, ls='--', lw=0.7, color='k')

plt.tight_layout()
plt.savefig(OUT / "oscillators.png", dpi=150, bbox_inches='tight')
plt.show()


---
## 6. Sanity Checks

| Test | Criterion | Significance |
|------|-----------|-------------|
| Energy drift (harmonic, no damping) | $< 10^{-6}$ relative over 60 time units | RK4 quality check |
| Period (harmonic) | $T = 2\pi$ within $0.1\%$ | Analytic comparison |
| Damped: $x \to 0$ | $|x(t_{\max})| < 0.01$ for strong damping | Attractor at origin |
| Phase portrait closure (conservative) | End point $\approx$ start point | Periodicity |


In [ ]:
print("=" * 55)
print("SANITY CHECKS — Project 03.1 Oscillators")
print("=" * 55)

# 1. Energy conservation (harmonic, no damping)
t, x, v = integrate_oscillator([1.0, 0.0], t_max=100.0, dt=0.01,
                                params={"k":1.0,"power":2.0})
E = 0.5*v**2 + 0.5*x**2
drift = np.max(np.abs(E - E[0])) / E[0]
ok = drift < 1e-5
print(f"\n1. Harmonic energy drift (100 time units): {drift:.2e}  {'✓' if ok else '✗'}")

# 2. Period check
x_sign = np.sign(x)
crossings = np.where(np.diff(x_sign) > 0)[0]   # positive zero crossings
if len(crossings) >= 2:
    T_numerical = np.mean(np.diff(t[crossings]))
    T_theory    = 2 * np.pi
    period_err  = abs(T_numerical - T_theory) / T_theory
    ok2 = period_err < 0.001
    print(f"2. Period: T_num={T_numerical:.5f}, T_theory={T_theory:.5f}, "
          f"err={period_err*100:.3f}%  {'✓' if ok2 else '✗'}")

# 3. Damped oscillator convergence
t, x_d, v_d = integrate_oscillator([1.5, 0.0], t_max=60.0, dt=0.01,
                                   params={"k":1.0,"power":2.0,"b":0.5})
final_amp = np.max(np.abs(x_d[-100:]))
ok3 = final_amp < 0.01
print(f"3. Damped: final amplitude = {final_amp:.4f}  {'✓' if ok3 else '✗'}")

print("\n" + "=" * 55)
